### Online Retail — ELT/ETL Project

A data pipeline project using the [UCI Online Retail dataset](https://archive.ics.uci.edu/dataset/352/online+retail)
(541,909 transactions, UK-based online retailer).

#### **Goal**
Practice ETL and ELT pipeline patterns end-to-end, using SQL as the primary transform
layer (ELT) with a smaller Python/pandas branch (ETL) for comparison. The analysis
builds toward RFM (Recency, Frequency, Monetary) customer segmentation.

#### **Stack**
- Python (extract, load, EDA support)
- PostgreSQL via Supabase (transforms, storage)
- DBeaver (SQL exploration)



#### 1. Setup

In [1]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sqlalchemy import create_engine, URL
from dotenv import load_dotenv
import os

load_dotenv()

True

#### 2. Extract

Pull the dataset directly from the UCI Machine Learning Repository.

In [2]:
online_retail = fetch_ucirepo(id=352) 

df = online_retail.data.original 
df.shape

(541909, 8)

In [3]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


#### 3. Load raw data to Postgres (Supabase)

Loaded as-is, no cleaning applied yet — this preserves a raw source of truth for the
ELT pipeline, so any transform step can be re-run without re-extracting.

In [4]:
db_url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USERNAME"),
    password=os.getenv("SUPABASE_DB_PASSWORD"),
    host="aws-1-eu-west-1.pooler.supabase.com",  
    port=5432,
    database="postgres",
)
engine = create_engine(db_url)

In [5]:
df.to_sql("retail_orders", engine, if_exists="replace", index=False)

909

In [6]:
pd.read_sql("SELECT COUNT(*) FROM retail_orders", engine)

,count
0,541909
